# Graph End-to-End Validation Notebook
Este notebook executa os testes de uso das classes novas em sequência:
1. Carrega `.env` e autentica no Graph
2. Obtém e apresenta o conteúdo do site
3. Lista arquivos do drive e salva em DataFrame (`df_drive_items`)
4. Executa fluxo de arquivo: write, update e load
5. Executa testes de listas (views, colunas, itens, create e update)

In [ ]:
from __future__ import annotations
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Garante carregamento do .env a partir da raiz do repositório
repo_root = Path.cwd()
if not (repo_root / '.env').exists() and (repo_root.parent / '.env').exists():
    repo_root = repo_root.parent

env_path = repo_root / '.env'
load_dotenv(env_path)

print(f'.env carregado de: {env_path}')
print('Pronto para iniciar os testes.')

In [ ]:
from msgraphtest.auth import GraphClient
from msgraphtest.drive import GraphDrive
from msgraphtest.lists import GraphList

client = GraphClient()
auth = client.authenticator
print('Cliente Graph inicializado com sucesso.')

site_attributes = {
    'sharepoint_site_id': auth.sharepoint_site_id,
    'site_graph_id': auth.site_graph_id,
    'site_name': auth.site_name,
    'site_display_name': auth.site_display_name,
    'site_web_url': auth.site_web_url,
}

print('\nAtributos do site conectado:')
for key, value in site_attributes.items():
    print(f'- {key}: {value}')

In [ ]:
site_contents = auth.get_site_contents()
site_data = site_contents.get('site', {})
drives_data = site_contents.get('drives', [])
lists_data = site_contents.get('lists', [])

print('Resumo do conteúdo do site:')
print(f"- Site: {site_data.get('displayName', site_data.get('name', '-'))}")
print(f"- Drives encontrados: {len(drives_data)}")
print(f"- Lists encontradas: {len(lists_data)}")

df_site_drives = pd.json_normalize(drives_data) if drives_data else pd.DataFrame()
df_site_lists = pd.json_normalize(lists_data) if lists_data else pd.DataFrame()

print('\nDrives Disponíveis:')
display(df_site_drives.head())

print('Lists Disponíveis:')
display(df_site_lists.head())

In [ ]:
drive = GraphDrive(client=client)
drive_items = drive.list_drive_items()

df_drive_items = pd.json_normalize(drive_items) if drive_items else pd.DataFrame()
num_files = len([item for item in drive_items if 'folder' not in item])

print(f'Total de itens no drive root: {len(drive_items)}')
print(f'Total de arquivos (sem pastas): {num_files}')
print('\nDados disponíveis:')

display(df_drive_items.head())

## File Write, Update and Load
Este bloco cria (ou reutiliza) um arquivo de teste, atualiza o conteúdo e lê novamente para validação.

In [ ]:
test_local_file = repo_root / 'notebooks' / 'downloads' / 'notebook_test_upload.txt'
test_local_file.parent.mkdir(parents=True, exist_ok=True)
test_local_file.write_text(
    f'Notebook initial content - {datetime.now(timezone.utc).isoformat()}\\n',
    encoding='utf-8',
)

upload_result = drive.upload_file(
    local_path=test_local_file,
    remote_folder='root',
    remote_name=f'notebook_test_upload_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}.txt',
)
target_item_id = str(upload_result.get('id', ''))

if not target_item_id:
    raise RuntimeError('Falha ao obter item id após upload do arquivo de teste.')

print('Write concluído (upload):')
print(f"- Item ID: {target_item_id}")
print(f"- Nome: {upload_result.get('name')}")

updated_content = (
    f'Notebook updated content - {datetime.now(timezone.utc).isoformat()}\\n'
    'Linha adicional para teste de update.\\n'
    'Fim.\\n'
 )
update_result = drive.write_file_content(target_item_id, updated_content)

print('Update concluído:')
print(f"- Item ID retornado: {update_result.get('id')}")

loaded_content = drive.read_file_content(target_item_id)
print('Load concluído (conteúdo atual lido):')
print(loaded_content)

download_path = repo_root / 'notebooks' / 'downloads' / f'downloaded_{target_item_id}.txt'
saved_path = drive.download_file(target_item_id, download_path)
print(f'Arquivo carregado para disco local em: {saved_path}')

## List Tests
Este bloco executa os testes de lista: views, colunas, itens, create e update.

In [ ]:
list_client = GraphList(client=client)

list_views = list_client.get_list_views()
list_columns = list_client.get_list_columns()
list_items = list_client.get_list_items(
    include_title=True,
    fields_only=True,
    include_item_id=True,
)

print(f'Views encontradas: {len(list_views)}')
print(f'Colunas consultadas (filtradas): {len(list_columns)}')
print(f'Itens encontrados: {len(list_items)}')


column_rename_map = {
    col["name"]: col.get("displayName", col["name"])
    for col in list_columns
    if col.get("name")
}
column_rename_map["id"] = "Item ID"
df_list_items = pd.DataFrame(list_items).rename(columns=column_rename_map) if list_items else pd.DataFrame()

print('DataFrame `df_list_items` criado (sem metadados internos, mantendo o Item ID).')
display(df_list_items.head())

test_title = f"Notebook item {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}"
created_item = list_client.create_list_item({'Title': test_title})
created_item_id = str(created_item.get('id', ''))

if not created_item_id:
    raise RuntimeError('Falha ao obter o ID do item criado na lista.')

print('Create de item concluído:')
print(f'- Item ID: {created_item_id}')
print(f"- Título inicial: {test_title}")

updated_title = f"{test_title} (updated)"
update_result = list_client.update_list_item(created_item_id, {'Title': updated_title})

print('Update de item concluído:')
print(update_result)

refreshed_items = list_client.get_list_items(
    include_title=True,
    fields_only=True,
    include_item_id=True,
)
df_list_items_after_update = pd.DataFrame(refreshed_items).rename(columns=column_rename_map) if refreshed_items else pd.DataFrame()
print(f'Total de itens após create/update: {len(refreshed_items)}')
display(df_list_items_after_update.head())